# 01 knowledge corpus

**Owner:** Daniel. Original notebook cells: 66, 67.

Run in the shared Databricks session using `run_pipeline.ipynb` or the documented stage runner. This is not a standalone clean-kernel notebook. Saved outputs have been removed for public-repository safety. See `docs/execution_order.md`.

### Build & Persist Approved Non-PHI Optometry RAG Corpus + Manifest

In [ ]:
# Approved Non-PHI Optometry RAG Corpus & Metadata Manifest

import os
import json
import pandas as pd

TEAM_VOLUME = "/Volumes/workspace/default/eye_style_team2"
RAG_DATA_DIR = f"{TEAM_VOLUME}/rag"
CORPUS_DIR = f"{RAG_DATA_DIR}/corpus"
os.makedirs(CORPUS_DIR, exist_ok=True)

RAG_DOCUMENTS = [
    {
        "doc_id": "DOC-01",
        "source_title": "Eye Style Scheduling & Appointment Readiness SOP",
        "source_type": "Clinic SOP (Synthetic)",
        "version_date": "2026-10-15",
        "approved_for_mvp": True,
        "contains_phi": False,
        "sections": [
            {
                "section": "Appointment Durations & Booking Windows",
                "text": (
                    "Standard appointment durations at Eye Style Optometry are 45 minutes for a "
                    "Comprehensive Eye Exam, 30 minutes for a Contact Lens Fitting or Follow-Up, "
                    "20 minutes for a Medical Eye Follow-Up, and 30 minutes for Frame Styling and "
                    "Dispensing. Routine exams may be booked up to 90 days in advance. Two same-day "
                    "urgent slots (10:30 AM and 3:30 PM) are reserved each clinic day for red-eye, "
                    "sudden vision changes, or broken eyewear emergencies. Front-desk staff must not "
                    "double-book optometrist exam lanes without explicit Practice Owner or doctor approval."
                ),
            },
            {
                "section": "Pre-Visit Readiness & Cancellation Policy",
                "text": (
                    "Front-desk staff perform an appointment-readiness review 2 business days before "
                    "each scheduled visit. Readiness requires: (1) confirmed appointment slot and visit "
                    "type, (2) verified vision or medical insurance status with no open exceptions, and "
                    "(3) updated patient contact preferences. Patients are requested to provide at least "
                    "24 hours notice to cancel or reschedule. Eye Style Optometry does not automatically "
                    "cancel appointments or assess automated no-show fees through the AI platform; any "
                    "schedule change requires manual staff confirmation in Crystal PM."
                ),
            },
        ],
    },
    {
        "doc_id": "DOC-02",
        "source_title": "No-Show Risk Triage & Follow-Up Protocol",
        "source_type": "Clinic SOP (Synthetic)",
        "version_date": "2026-11-01",
        "approved_for_mvp": True,
        "contains_phi": False,
        "sections": [
            {
                "section": "Interpreting Booking-Time Model Risk Scores",
                "text": (
                    "The Scheduling Agent incorporates a booking-time neural-network no-show model "
                    "(eye_style_mlp_v1) that outputs an uncalibrated classification risk score between "
                    "0 and 1. Staff must interpret this value strictly as a relative risk score for "
                    "triage—never as a calibrated probability or percentage chance that a patient will "
                    "miss an appointment. Appointments with a model risk score at or above the fixed "
                    "0.50 classification threshold are flagged as High priority for human review. "
                    "Appointments below 0.50 are assigned Routine priority. Staff must not independently "
                    "alter the 0.50 model threshold."
                ),
            },
            {
                "section": "Human-in-the-Loop Outreach Cadence",
                "text": (
                    "For High-priority appointments flagged by the Scheduling Agent, front-desk staff "
                    "should review the booking lead time and verification status and place a manual "
                    "confirmation call 48 hours prior to the visit, followed by an approved reminder "
                    "check at 24 hours if unconfirmed. For Routine-priority appointments, standard "
                    "reminder workflow applies. The AI assistant is strictly read-only and recommendation-based: "
                    "it cannot automatically send SMS messages, place calls, or modify appointments."
                ),
            },
        ],
    },
    {
        "doc_id": "DOC-03",
        "source_title": "Insurance Eligibility & Verification SOP",
        "source_type": "Clinic SOP (Synthetic)",
        "version_date": "2026-10-20",
        "approved_for_mvp": True,
        "contains_phi": False,
        "sections": [
            {
                "section": "Standard Pre-Visit Verification Timeline",
                "text": (
                    "Front-desk staff must verify insurance eligibility for all scheduled appointments "
                    "at least 2 business days (48 business hours) prior to the appointment date. "
                    "Verification confirms active plan status, exam copay, lens and frame material "
                    "allowances, contact lens fitting coverage, and benefit renewal frequency. "
                    "Because the capstone AI platform does not connect to live VSP, EyeMed, or payer "
                    "portals, all live eligibility checks must be performed directly by staff in the "
                    "authorized payer portal and recorded in Crystal PM."
                ),
            },
            {
                "section": "Handling Insurance Exceptions & Escalation",
                "text": (
                    "Any record marked 'Needs Review', 'Pending', or 'Expired' must be treated as "
                    "unresolved and never classified as Verified. Common exceptions include Member ID "
                    "mismatches, unconfirmed annual exam benefit eligibility, missing secondary medical "
                    "coverage details, or pending prior authorizations for medical follow-ups. When an "
                    "exception remains unresolved within 24 hours of the visit, front-desk staff must "
                    "call the patient to collect updated insurance card information or advise them of "
                    "self-pay options prior to check-in."
                ),
            },
        ],
    },
    {
        "doc_id": "DOC-04",
        "source_title": "Vision & Medical Insurance Payer Reference Guide",
        "source_type": "Payer Reference (Synthetic)",
        "version_date": "2026-10-18",
        "approved_for_mvp": True,
        "contains_phi": False,
        "sections": [
            {
                "section": "Vision Plans vs. Medical Insurance Routing",
                "text": (
                    "Eye Style Optometry accepts major vision plans including VSP Choice, EyeMed Insight, "
                    "Spectera, and Davis Vision for routine refractive eye exams and optical materials "
                    "(frames, lenses, or contact lenses). Medical eye visits—including diabetic retinal "
                    "evaluations, glaucoma monitoring, dry eye treatment, ocular infections, or sudden "
                    "flashes and floaters—must be verified and billed under the patient's medical "
                    "insurance (such as Medicare Medical or commercial major medical), not routine vision "
                    "plans. Front-desk staff should ask the primary reason for visit during scheduling."
                ),
            },
            {
                "section": "Coordination of Benefits & Authorizations",
                "text": (
                    "Routine vision benefits typically renew every 12 months for exams and lenses, and "
                    "every 12 to 24 months for frames depending on the employer tier. Patients cannot "
                    "combine full frame material allowances and elective contact lens allowances in the "
                    "same benefit cycle on standard VSP Choice or EyeMed Insight plans. Medical referrals "
                    "or HMO prior authorizations must be confirmed before rendering non-urgent specialty "
                    "diagnostic testing."
                ),
            },
        ],
    },
    {
        "doc_id": "DOC-05",
        "source_title": "Optical Order Fulfillment & Lab Tracking SOP",
        "source_type": "Clinic SOP (Synthetic)",
        "version_date": "2026-10-22",
        "approved_for_mvp": True,
        "contains_phi": False,
        "sections": [
            {
                "section": "Optical Order Status Definitions",
                "text": (
                    "Crystal-style optical frame and contact lens orders move through three core MVP "
                    "tracking statuses: 'InFulfillment' (order submitted to the optical lab or distributor "
                    "and awaiting receipt/inspection), 'ReadyForDispensing' (order received at clinic, "
                    "verified against the prescription, and ready for patient pickup), and 'Dispensed' "
                    "(eyewear fitted and handed to the patient with dispenseDate recorded). Staff and "
                    "agents must never fabricate receipt, lab, or dispensing events not present in records."
                ),
            },
            {
                "section": "Lab Turnaround Times & Overdue Escalation",
                "text": (
                    "Standard optical lab turnaround times with Pacific Optical Lab, Cascade Lens Works, "
                    "and Northwest RX Lab are 7 to 10 business days for single-vision stock lenses and "
                    "10 to 14 business days for digital progressives, high-index prescriptions, or "
                    "specialty anti-reflective/photochromic coatings. An InFulfillment order whose dueDate "
                    "has passed is classified as overdue. The optician must contact the assigned lab "
                    "first to obtain an updated ETA or remake status before contacting the patient."
                ),
            },
        ],
    },
    {
        "doc_id": "DOC-06",
        "source_title": "Optical Dispensing, Deposits & Pickup Follow-Up SOP",
        "source_type": "Clinic SOP (Synthetic)",
        "version_date": "2026-10-22",
        "approved_for_mvp": True,
        "contains_phi": False,
        "sections": [
            {
                "section": "Order Deposits & Pre-Dispense Quality Inspection",
                "text": (
                    "A minimum 50% deposit of the patient's estimated out-of-pocket responsibility is "
                    "required before submitting custom prescription frame orders to an optical lab. "
                    "Upon arrival from the lab, the optician must complete a quality inspection before "
                    "updating status to ReadyForDispensing: verify sphere/cylinder/axis on the lensometer, "
                    "check pupillary distance (PD) and segment height, inspect anti-reflective coatings "
                    "for scratches, and confirm frame alignment. Remaining balance is collected at dispense."
                ),
            },
            {
                "section": "Pickup Notification Cadence & Unclaimed Orders",
                "text": (
                    "Once an order passes quality inspection and is marked ReadyForDispensing, the optician "
                    "or front desk initiates manual patient notification within 1 business day (Day 1). "
                    "If the order is not picked up, follow-up outreach occurs on Day 7 and Day 14. "
                    "Orders remaining unclaimed after 30 days are escalated to the Practice Owner for "
                    "account review. All patient pickup outreach requires human staff approval."
                ),
            },
        ],
    },
    {
        "doc_id": "DOC-07",
        "source_title": "Contact Lens Order & Fitting Workflow",
        "source_type": "Clinic SOP (Synthetic)",
        "version_date": "2026-10-25",
        "approved_for_mvp": True,
        "contains_phi": False,
        "sections": [
            {
                "section": "Prescription Validity & Annual Evaluation",
                "text": (
                    "Contact lens orders require a valid, unexpired contact lens prescription specifying "
                    "brand, base curve, diameter, and power. Standard contact lens prescriptions expire "
                    "after 12 months and require an annual contact lens evaluation before renewal. "
                    "New wearers or patients changing lens brands must complete a trial-lens fitting "
                    "check with the optometrist before an annual supply order is finalized."
                ),
            },
            {
                "section": "Contact Lens Fulfillment & Tracking",
                "text": (
                    "Approved contact lens orders are placed through the authorized distributor and "
                    "typically arrive within 5 to 7 business days (specialty toric or multifocal lenses "
                    "may take up to 10 business days). Unlike frame orders, contact lens records in the "
                    "MVP synthetic schema do not include external optical lab or optician assignment "
                    "fields; staff track fulfillment via orderDate, dueDate, status, and dispenseDate."
                ),
            },
        ],
    },
    {
        "doc_id": "DOC-08",
        "source_title": "Front Desk & Optician Staff Onboarding Guide",
        "source_type": "Onboarding Guide (Synthetic)",
        "version_date": "2026-10-28",
        "approved_for_mvp": True,
        "contains_phi": False,
        "sections": [
            {
                "section": "Daily Opening & Closing Operational Checklists",
                "text": (
                    "During morning opening (8:00 AM - 8:30 AM), Front Desk reviews the Scheduling "
                    "and Insurance Verification dashboards for same-day readiness, High-priority no-show "
                    "flags, and unresolved insurance exceptions. Simultaneously, the Optician reviews "
                    "overdue InFulfillment frame orders and ReadyForDispensing pickup queues. Before "
                    "closing (5:00 PM), staff verify that all appointments scheduled 2 business days out "
                    "have been reviewed and that all lab check-ins are logged."
                ),
            },
            {
                "section": "Role Responsibilities & Owner Escalation",
                "text": (
                    "Front Desk staff manage appointment scheduling, check-in/check-out, insurance "
                    "verification, and patient pickup coordination. Opticians manage frame styling, "
                    "measurements, lab tracking, quality inspection, and dispensing. The Practice Owner "
                    "oversees cross-workflow bottlenecks, unresolved high-priority queues, late lab "
                    "vendors, and policy exceptions. Staff should escalate repeated lab remakes, "
                    "unresolved payer disputes, or formal patient complaints to the Practice Owner."
                ),
            },
        ],
    },
    {
        "doc_id": "DOC-09",
        "source_title": "AI Assistant Governance, Role Boundaries & Non-PHI Policy",
        "source_type": "Governance Policy (Synthetic)",
        "version_date": "2026-11-05",
        "approved_for_mvp": True,
        "contains_phi": False,
        "sections": [
            {
                "section": "Role-Based Access & Scoped Specialist Agents",
                "text": (
                    "The Eye Style AI Operations Platform enforces server-side least-privilege access "
                    "across three roles: Owner (Scheduling, Insurance Verification, Optical Orders, RAG), "
                    "Front Desk / Receptionist (Scheduling, Insurance Verification, Optical Orders, RAG), "
                    "and Optician (Optical Orders and RAG only). Opticians are denied access to Scheduling "
                    "and Insurance Verification agents. Billing automation, insurance receivables, clinical "
                    "notes, and record writes are outside the protected capstone MVP and strictly blocked."
                ),
            },
            {
                "section": "Read-Only Safeguards, Abstention & Non-PHI Boundaries",
                "text": (
                    "All capstone agents are read-only and recommendation-based. Agents must never "
                    "modify appointments, create invoices, contact patients autonomously, or offer "
                    "medical diagnoses. The RAG knowledge assistant must answer strictly from approved "
                    "non-PHI corpus documents with explicit source citations. If the corpus lacks "
                    "sufficient supporting evidence, the RAG assistant must abstain and state that "
                    "there is insufficient evidence rather than speculating. Live patient PHI and "
                    "production Crystal PM API connections are prohibited during the capstone."
                ),
            },
        ],
    },
    {
        "doc_id": "DOC-10",
        "source_title": "Patient Communication & Clinical Escalation Standards",
        "source_type": "Clinic SOP (Synthetic)",
        "version_date": "2026-10-30",
        "approved_for_mvp": True,
        "contains_phi": False,
        "sections": [
            {
                "section": "Approved Outreach & Lab Delay Communication",
                "text": (
                    "When communicating an optical lab delay, staff must first confirm an updated ETA "
                    "with the lab, then proactively contact the patient via their preferred channel "
                    "with an apology, the revised expected arrival date, and an option to schedule a "
                    "dispensing fit once verified. Staff must never promise same-day delivery without "
                    "physical receipt and lensometry verification. AI-drafted reminders or follow-ups "
                    "always require human staff review and manual sending."
                ),
            },
            {
                "section": "Clinical Symptom & Urgent Medical Escalation",
                "text": (
                    "Non-clinical staff (Front Desk and Opticians) and the AI assistant must never "
                    "diagnose ocular conditions, recommend prescription medications, or triage clinical "
                    "severity independently. Any patient reporting sudden vision loss, new flashes or "
                    "floaters, severe eye pain, chemical exposure, or ocular trauma must be escalated "
                    "immediately to the on-duty optometrist—or directed to emergency medical care if "
                    "after clinic hours."
                ),
            },
        ],
    },
]

# Persist individual documents and manifest to Unity Catalog Volume
manifest_records = []
for doc in RAG_DOCUMENTS:
    doc_path = f"{CORPUS_DIR}/{doc['doc_id']}.json"
    with open(doc_path, "w", encoding="utf-8") as f:
        json.dump(doc, f, indent=2)

    manifest_records.append({
        "doc_id": doc["doc_id"],
        "source_title": doc["source_title"],
        "source_type": doc["source_type"],
        "version_date": doc["version_date"],
        "section_count": len(doc["sections"]),
        "sections": [s["section"] for s in doc["sections"]],
        "approved_for_mvp": doc["approved_for_mvp"],
        "contains_phi": doc["contains_phi"],
        "file_path": doc_path,
    })

corpus_manifest = {
    "corpus_version": "eye_style_rag_corpus_v1",
    "document_count": len(RAG_DOCUMENTS),
    "section_count": sum(len(d["sections"]) for d in RAG_DOCUMENTS),
    "contains_phi": False,
    "documents": manifest_records,
}

manifest_path = f"{RAG_DATA_DIR}/rag_corpus_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(corpus_manifest, f, indent=2)

print("=== RAG CORPUS & MANIFEST SAVED ===")
print(f"Directory: {RAG_DATA_DIR}")
print(f"Documents: {corpus_manifest['document_count']} | Sections: {corpus_manifest['section_count']}")
display(pd.DataFrame(manifest_records)[["doc_id", "source_title", "source_type", "version_date", "section_count", "contains_phi"]])